# T3 — Experimentos II

Experimentos E4, E5, E6 y E7, con P13 y hallazgo.

## E4 — Dependencia del reparto entrenamiento/prueba

**Hipótesis:** creo que el resultado será bastante parecido aunque cambie la semilla del reparto, porque el modelo utiliza las mismas características y los datos mantienen una estructura similar.

**Cambio:** únicamente cambio la semilla utilizada para separar los datos de entrenamiento y prueba.

In [ ]:
resultados = [evaluar_xy(X, y, semilla=s, mostrar=False)[2] for s in range(10)]
print("F1 en 10 repartos:", [round(r, 3) for r in resultados])
print(f"media={pd.Series(resultados).mean():.3f}  mínimo={min(resultados):.3f}  máximo={max(resultados):.3f})

**Resultado:** pendiente de ejecutar el notebook.

**Explicación:** el resultado puede cambiar dependiendo de qué IPs caen en entrenamiento y cuáles en prueba. Por tanto, un único reparto puede dar una impresión demasiado optimista o pesimista del modelo.

## E5 — Datos nuevos

**Hipótesis:** creo que el modelo mantendrá un rendimiento parecido con datos generados con otra semilla, aunque puede empeorar algo porque son datos que no ha utilizado durante el entrenamiento.

**Cambio:** se utilizan datos generados con otra semilla, manteniendo el modelo entrenado con los datos originales.

In [ ]:
try:
    X_b = construir_features(cargar_log("auth_b.log", verbose=False))
    y_b = cargar_etiquetas("etiquetas_b.csv", X_b.index)
    pred_b = modelo.predict(X_b[X.columns])
    print(confusion_matrix(y_b, pred_b))
    print(classification_report(y_b, pred_b, digits=3))
except FileNotFoundError:
    print("Primero genera auth_b.log y etiquetas_b.csv con el comando correspondiente.")

**Resultado:** pendiente de ejecutar el notebook.

**Explicación:** el modelo debe comprobarse con datos que no utilizó durante el entrenamiento para valorar si generaliza.

## E6 — Muy pocos atacantes

**Hipótesis:** creo que el recall será más inestable cuando haya muy pocos atacantes, porque al dividir los datos entre entrenamiento y prueba habrá pocos ejemplos de ataques disponibles.

**Cambio:** se reduce el número de atacantes hasta 12.

In [ ]:
try:
    X_p = construir_features(cargar_log("auth_poco.log", verbose=False))
    y_p = cargar_etiquetas("etiquetas_poco.csv", X_p.index)
    print(f"Atacantes: {int(y_p.sum())} de {len(y_p)} IPs ({y_p.mean():.0%})")
    evaluar_xy(X_p, y_p)
    recalls = [evaluar_xy(X_p, y_p, semilla=s, mostrar=False)[1] for s in range(10)]
    print("recall en 10 repartos:", [round(r, 2) for r in recalls])
except FileNotFoundError:
    print("Primero genera auth_poco.log y etiquetas_poco.csv con el comando correspondiente.")

**Resultado:** pendiente de ejecutar el notebook.

**Explicación:** cuando hay pocos ejemplos de la clase atacante, cada ejemplo tiene mucho peso en el resultado y el recall puede variar bastante entre repartos.

## E7 — Detectar sin etiquetas

**Hipótesis:** creo que un modelo sin etiquetas podrá detectar algunas IPs atacantes porque busca comportamientos que se salen de lo normal, aunque no sabrá necesariamente distinguir todos los ataques.

**Cambio:** se utiliza IsolationForest sin utilizar las etiquetas para aprender.

In [ ]:
CONTAMINACION = 0.25
cols = ["n_eventos", "n_fallos", "n_usuarios", "n_inv", "fallos_por_min", "max_fallos_5min", "ratio_fallos"]
Z = StandardScaler().fit_transform(X[cols])
iso = IsolationForest(contamination=CONTAMINACION, random_state=42)
anomalo = (iso.fit_predict(Z) == -1).astype(int)
print(confusion_matrix(y, anomalo))
print(classification_report(y, anomalo, digits=3))

In [ ]:
for contaminacion in [0.05, 0.1, 0.25, 0.4]:
    iso = IsolationForest(contamination=contaminacion, random_state=42)
    anomalo = (iso.fit_predict(Z) == -1).astype(int)
    print(f"\nContaminación: {contaminacion}")
    print(confusion_matrix(y, anomalo))
    print(classification_report(y, anomalo, digits=3))

**Resultado:** pendiente de ejecutar el notebook.

**Explicación:** IsolationForest busca comportamientos anómalos sin utilizar etiquetas, pero el resultado depende del valor de contaminación elegido.

## P13 — ¿Qué experimento me ha enseñado algo inesperado?

El experimento que más me ha enseñado ha sido E6. Al reducir mucho el número de atacantes, el recall puede cambiar bastante dependiendo del reparto de los datos. Esto significa que un modelo puede parecer muy bueno mirando solamente la precisión, pero ese número no cuenta toda la historia.

Por eso no deberíamos confiar en un único número de acierto. Hay que mirar también precision, recall, F1 y los falsos positivos y negativos, además de comprobar qué ocurre cuando cambiamos los datos.

## Hallazgo

E4 muestra que el F1 puede cambiar al modificar el reparto entre entrenamiento y prueba. E6 permite comprobar si este efecto aumenta cuando hay muy pocos atacantes. Por tanto, una única medida de precisión no es suficiente para decidir si el detector es fiable.